# 01. Camada de dados (DAL)

Desenvolve as funções de acesso a dados (baixar, gravar/ler SQLite, retornos). **F1, NF6.**

In [1]:
import sys, os, tempfile
_cwd = os.getcwd() 
RAIZ = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'tests' else _cwd 
if RAIZ not in sys.path: 
    sys.path.insert(0, RAIZ)
import sqlite3
from contextlib import closing
from typing import Sequence
import pandas as pd

## Desenvolvimento

As funções abaixo baixam os dados, gravam e leem no SQLite e calculam os retornos. Foram escritas e testadas aqui, e depois passaram para app/dal.py. Requisitos F1 e NF6.

In [ ]:
# frequencias que a ingestao aceita. O formato da coluna data acompanha:
# AAAA-MM no mensal e AAAA-MM-DD no diario.
FORMATO_DATA = {"1mo": "%Y-%m", "1d": "%Y-%m-%d"}

# qual serie do CDI pedir pra API do Banco Central em cada frequencia.
# A 4391 e o CDI acumulado no mes; a 12 e o CDI do dia.
_SERIE_CDI = {"1mo": 4391, "1d": 12}

# a API recusa pedido de serie diaria com mais de 10 anos (responde 406).
# No mensal nao tem esse limite. Periodo maior que isso e baixado em pedacos.
_LIMITE_ANOS_SGS = {"1mo": None, "1d": 10}

# essa API demora bastante em janela diaria grande,
# entao o tempo de espera aqui e bem maior que o do Yahoo.
_TIMEOUT_SGS = 90

# O esquema das tres tabelas do projeto: a data e chave primaria (AAAA-MM tem
# 7 caracteres e AAAA-MM-DD tem 10), nenhuma coluna aceita nulo, o fechamento e
# positivo e o CDI nao e negativo.
ESQUEMAS: dict[str, tuple[str, ...]] = {
    "ibovespa": (
        "data TEXT PRIMARY KEY NOT NULL CHECK (length(data) IN (7, 10))",
        "fechamento REAL NOT NULL CHECK (fechamento > 0)",
    ),
    "cdi": (
        "data TEXT PRIMARY KEY NOT NULL CHECK (length(data) IN (7, 10))",
        "cdi REAL NOT NULL CHECK (cdi >= 0)",
    ),
    "retornos": (
        "data TEXT PRIMARY KEY NOT NULL CHECK (length(data) IN (7, 10))",
        "ibov REAL NOT NULL",
        "cdi REAL NOT NULL",
    ),
}

In [3]:

def _validar_identificador(nome: str) -> None:
    """Confere se o nome da tabela e valido antes de montar o SQL."""
    if not nome.isidentifier():
        raise ValueError(f"nome de tabela invalido: {nome!r}")

In [4]:

def _formato_data(frequencia: str) -> str:
    """Formato da coluna data para a frequencia pedida (valida o argumento)."""
    try:
        return FORMATO_DATA[frequencia]
    except KeyError:
        raise ValueError(
            f"frequencia desconhecida: {frequencia!r} (use {sorted(FORMATO_DATA)})."
        ) from None

In [ ]:
def baixar_precos(
    ativos: Sequence[str],
    inicio: str,
    fim: str | None = None,
    frequencia: str = "1mo",
) -> pd.DataFrame:
    """
    Baixa os precos de fechamento no Yahoo Finance. (F1)

    Recebe a lista de tickers (por exemplo ["^BVSP"]), a data inicial e a
    final no formato AAAA-MM-DD (se a final vier None, usa hoje) e a
    frequencia, "1mo" pra mensal ou "1d" pra diario.

    Devolve um DataFrame com a coluna data na frente, no formato AAAA-MM ou
    AAAA-MM-DD conforme a frequencia, e uma coluna por ticker.

    O campo lido e o 'close' da API, o fechamento sem ajuste. Para o ^BVSP da
    no mesmo, porque indice de pontos nao paga dividendo nem desdobra.

    Usa a API publica de graficos do Yahoo com o urllib, que ja vem no Python.
    Cheguei aqui porque o yfinance e o curl_cffi davam erro de certificado SSL
    no Windows. Os imports ficam dentro da funcao pra que os testes de banco e
    de retorno continuem rodando sem internet.
    """
    import json
    from datetime import datetime, timezone
    from urllib.parse import quote
    from urllib.request import Request, urlopen

    fmt = _formato_data(frequencia)

    def _epoch(d: str) -> int:
        return int(pd.Timestamp(d, tz="UTC").timestamp())

    p1 = _epoch(inicio)
    p2 = _epoch(fim or datetime.now(timezone.utc).strftime("%Y-%m-%d"))

    series: dict[str, pd.Series] = {}
    for tk in ativos:
        url = (f"https://query1.finance.yahoo.com/v8/finance/chart/{quote(tk)}"
               f"?period1={p1}&period2={p2}&interval={frequencia}")
        req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(req, timeout=30) as resp:
            resposta = json.load(resp)
        resultado = (resposta.get("chart") or {}).get("result")
        if not resultado:
            raise ValueError(f"Yahoo nao retornou dados para {tk!r}.")
        res = resultado[0]
        ts = res.get("timestamp") or [] 
        close = res["indicators"]["quote"][0].get("close") or [] 
        datas = [datetime.fromtimestamp(t, tz=timezone.utc).strftime(fmt) for t in ts] 
        series[tk] = pd.Series(close, index=datas, name=tk) 

    df = pd.DataFrame(series).dropna(how="any") 
    df = df[~df.index.duplicated(keep="last")]
    return df.reset_index().rename(columns={"index": "data"})

In [6]:
def calcular_retornos(precos: pd.DataFrame, coluna_data: str = "data") -> pd.DataFrame:
    """Converte precos/niveis em retornos simples por periodo. (F1)

    A primeira observacao e descartada, porque nao ha retorno anterior, entao
    sobram T - 1 linhas. Toda coluna que nao seja a coluna_data e tratada como
    serie de preco.

    O CDI nao passa por aqui, porque ele ja e um retorno e nao um preco. Ele
    vai direto pra coluna cdi da tabela retornos.
    """
    if coluna_data not in precos.columns:
        raise KeyError(f"coluna '{coluna_data}' ausente em precos.")

    datas = precos[coluna_data].iloc[1:].to_numpy()
    numericas = precos.drop(columns=[coluna_data])
    ret = numericas.pct_change(fill_method=None).iloc[1:].reset_index(drop=True)
    ret.insert(0, coluna_data, datas)
    return ret

In [ ]:
def _tabela_existe(con: sqlite3.Connection, tabela: str) -> bool:
    """Diz se a tabela ja existe no banco aberto em con."""
    achou = con.execute( 
        "SELECT 1 FROM sqlite_master WHERE type = 'table' AND name = ?", (tabela,) 
    ).fetchone() 
    return achou is not None 

In [ ]:
def _erro_original_sqlite(exc: Exception) -> None: 
    """Devolve o erro original do SQLite, que o pandas esconde dentro do dele."""
    if isinstance(exc.__cause__, sqlite3.Error): 
        raise exc.__cause__ from None 
    raise exc 

In [ ]:
def gravar_sqlite(
    df: pd.DataFrame,
    db_path: str,
    tabela: str,
    if_exists: str = "replace",
) -> None:
    """Grava um DataFrame numa tabela do SQLite. (F1, NF6)

    Se a tabela for uma das tres do projeto, ela e criada a partir do
    ESQUEMAS, com chave primaria na data, NOT NULL em tudo e os CHECK de
    dominio. Data repetida, celula vazia ou fechamento negativo param aqui,
    com IntegrityError. Qualquer outra tabela cai no caminho normal do pandas,
    sem restricao nenhuma.

    A gravacao e tudo ou nada. O "replace" monta a tabela nova ao lado da
    antiga e so troca as duas no fim, entao uma linha que bata num CHECK deixa
    a base que ja estava no banco intacta.
    """
    _validar_identificador(tabela)
    colunas = ESQUEMAS.get(tabela) 
    with closing(sqlite3.connect(db_path)) as con:
        try:
            if colunas is None: 
                df.to_sql(tabela, con, if_exists=if_exists, index=False) 
                con.commit() 
                return 
            if if_exists == "fail" and _tabela_existe(con, tabela): 
                raise ValueError(f"a tabela {tabela!r} ja existe em {db_path}.") 
            if if_exists != "replace": 
                con.execute(f"CREATE TABLE IF NOT EXISTS {tabela} ({', '.join(colunas)})") 
                df.to_sql(tabela, con, if_exists="append", index=False) 
                con.commit()
                return

            provisoria = f"{tabela}__novo" 
            con.execute(f"DROP TABLE IF EXISTS {provisoria}") 
            con.execute(f"CREATE TABLE {provisoria} ({', '.join(colunas)})") 
            try:
                df.to_sql(provisoria, con, if_exists="append", index=False) 
                con.execute(f"DROP TABLE IF EXISTS {tabela}")
                con.execute(f"ALTER TABLE {provisoria} RENAME TO {tabela}")
                con.commit()
            except Exception: 
                con.rollback()
                con.execute(f"DROP TABLE IF EXISTS {provisoria}")
                con.commit()
                raise 
        except Exception as exc: 
            _erro_original_sqlite(exc)

In [10]:
def ler_sqlite(db_path: str, tabela: str) -> pd.DataFrame:
    """Le uma tabela do banco SQLite para um DataFrame. (F1, NF6)"""
    _validar_identificador(tabela)
    with closing(sqlite3.connect(db_path)) as con:
        return pd.read_sql(f"SELECT * FROM {tabela}", con)

In [11]:
def _janelas(inicio: pd.Timestamp, fim: pd.Timestamp, limite_anos: int | None):
    """Corta o periodo em pedacos de no maximo limite_anos.

    Com limite_anos None devolve um pedaco so.
    """
    if limite_anos is None or fim <= inicio + pd.DateOffset(years=limite_anos):
        return [(inicio, fim)]
    partes, atual = [], inicio
    while atual <= fim:
        prox = min(atual + pd.DateOffset(years=limite_anos), fim)
        partes.append((atual, prox))
        atual = prox + pd.Timedelta(days=1)
    return partes

In [ ]:
def baixar_cdi_bcb(inicio: str, fim: str | None = None,
                   frequencia: str = "1mo") -> pd.DataFrame:
    """
    Baixa o CDI da API do Banco Central. (F1)

    No mensal pede a serie 4391, que e o CDI acumulado no mes; no diario pede a
    serie 12, que e o CDI do dia. Devolve um DataFrame com data e cdi, com o
    cdi ja em decimal e por periodo (uns 0.0034 no mensal e 0.00047 no diario).
    A fonte e oficial e de graca, nao precisa cadastro, mas precisa internet.

    Os imports ficam dentro da funcao e sao todos da biblioteca padrao. A serie
    diaria vem em pedacos de 10 anos, que e o limite da API, e depois eles sao
    juntados.
    """
    import json
    from datetime import date as _date
    from urllib.request import urlopen

    fmt = _formato_data(frequencia)
    serie = _SERIE_CDI[frequencia]
    ini = pd.to_datetime(inicio)
    dfim = pd.to_datetime(fim or _date.today().isoformat())

    partes = []
    for janela_ini, janela_fim in _janelas(ini, dfim, _LIMITE_ANOS_SGS[frequencia]):
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie}/dados"
               f"?formato=json&dataInicial={janela_ini.strftime('%d/%m/%Y')}"
               f"&dataFinal={janela_fim.strftime('%d/%m/%Y')}")
        with urlopen(url, timeout=_TIMEOUT_SGS) as resp:
            partes.extend(json.load(resp)) 
    if not partes:
        raise ValueError("BCB nao retornou CDI para o periodo pedido.")

    df = pd.DataFrame(partes)
    df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y").dt.strftime(fmt)
    df["cdi"] = df["valor"].astype(float) / 100.0   # % por periodo -> decimal
    # Janelas consecutivas podem repetir a data de fronteira.
    return df[["data", "cdi"]].drop_duplicates(subset="data").reset_index(drop=True)

**Teste**: os retornos, a gravação e leitura no SQLite (grava e lê de volta) e os downloads de verdade, que ficam protegidos por try.

In [13]:

import pandas as pd, os, tempfile
precos = pd.DataFrame({'data': ['2000-01','2000-02','2000-03','2000-04'],
                       'ibov': [100.,102.,99.,105.]})

print('calcular_retornos:')
print(calcular_retornos(precos))

calcular_retornos:
      data      ibov
0  2000-02  0.020000
1  2000-03 -0.029412
2  2000-04  0.060606


In [14]:
esp = precos['ibov'].pct_change(fill_method=None).iloc[1:].reset_index(drop=True)
assert calcular_retornos(precos)['ibov'].round(10).tolist() == esp.round(10).tolist()

In [15]:
# a tabela 'ibovespa' tem esquema proprio (data + fechamento), entao o
# DataFrame precisa trazer essas colunas
niveis = precos.rename(columns={'ibov': 'fechamento'})
db = os.path.join(tempfile.gettempdir(), 'dev01.db')
if os.path.exists(db):
    os.remove(db)
gravar_sqlite(niveis, db, 'ibovespa')
lido = ler_sqlite(db, 'ibovespa')
pd.testing.assert_frame_equal(niveis.reset_index(drop=True), lido, check_dtype=False)

**Teste**: as restrições do ESQUEMAS. Base suja tem que ser recusada com IntegrityError, e a base que já estava no banco tem que sobreviver à recusa.

In [16]:
with closing(sqlite3.connect(db)) as con:
    print(con.execute(
        "SELECT sql FROM sqlite_master WHERE name = 'ibovespa'").fetchone()[0])

CREATE TABLE "ibovespa" (data TEXT PRIMARY KEY NOT NULL CHECK (length(data) IN (7, 10)), fechamento REAL NOT NULL CHECK (fechamento > 0))


In [17]:
sujas = {
    'data repetida':pd.DataFrame({'data': ['2000-01', '2000-01'], 'fechamento': [1., 2.]}),
    'fechamento <= 0': pd.DataFrame({'data': ['2000-09'], 'fechamento': [0.]}),
    'celula vazia': pd.DataFrame({'data': ['2000-09'], 'fechamento': [None]}),
    'data fora do formato': pd.DataFrame({'data': ['jan/2000'], 'fechamento': [1.]}),
}
recusadas = []
for nome, ruim in sujas.items():
    try:
        gravar_sqlite(ruim, db, 'ibovespa')
    except sqlite3.IntegrityError:
        recusadas.append(nome)
        print('recusado: %-22s -> base intacta com %d linhas'
              % (nome, len(ler_sqlite(db, 'ibovespa'))))

recusado: data repetida          -> base intacta com 4 linhas
recusado: fechamento <= 0        -> base intacta com 4 linhas
recusado: celula vazia           -> base intacta com 4 linhas
recusado: data fora do formato   -> base intacta com 4 linhas


In [18]:
assert recusadas == list(sujas), 'passaram sem recusa: %s' % (set(sujas) - set(recusadas))
assert len(ler_sqlite(db, 'ibovespa')) == len(niveis)

In [19]:
# uma tabela fora do ESQUEMAS continua no caminho generico do pandas, sem restricao
gravar_sqlite(pd.DataFrame({'x': [1, 1]}), db, 'tabela_livre')
assert len(ler_sqlite(db, 'tabela_livre')) == 2

In [20]:
try:
    px = baixar_precos(['^BVSP'], '2023-01-01', '2023-04-01')
    print(px)
    print(baixar_cdi_bcb('2023-01-01', '2023-04-01'))

    hoje = baixar_precos(['^BVSP'], '2000-01-01')
    assert hoje['data'].is_unique, hoje['data'][hoje['data'].duplicated()].tolist()
    print('datas unicas em', len(hoje), 'meses ate hoje: ok')
except Exception as e:
    print('download offline:', type(e).__name__, e)

      data     ^BVSP
0  2023-01  113532.0
1  2023-02  104932.0
2  2023-03  101882.0
      data     cdi
0  2023-01  0.0112
1  2023-02  0.0092
2  2023-03  0.0117
3  2023-04  0.0092


datas unicas em 321 meses ate hoje: ok


In [21]:
# --- frequencia e fatiamento de janelas (offline, deterministico) ---
from app import dal as _dal

assert _dal.FORMATO_DATA == {'1mo': '%Y-%m', '1d': '%Y-%m-%d'}
assert _dal._formato_data('1mo') == '%Y-%m' and _dal._formato_data('1d') == '%Y-%m-%d'

In [22]:
# a SGS recusa (HTTP 406) series diarias de mais de 10 anos por requisicao
j_curta = _dal._janelas(pd.Timestamp('2022-05-22'), pd.Timestamp('2026-08-06'), 10)
j_longa = _dal._janelas(pd.Timestamp('2000-01-01'), pd.Timestamp('2026-08-06'), 10)
j_sem   = _dal._janelas(pd.Timestamp('2000-01-01'), pd.Timestamp('2026-08-06'), None)

print('janelas: curta=%d longa=%d sem_limite=%d' % (len(j_curta), len(j_longa), len(j_sem)))

janelas: curta=1 longa=3 sem_limite=1


In [23]:
assert len(j_curta) == 1 and len(j_sem) == 1 and len(j_longa) == 3
assert j_longa[0][0] == pd.Timestamp('2000-01-01') and j_longa[-1][1] == pd.Timestamp('2026-08-06')
assert all(a <= b for a, b in j_longa)
assert all(j_longa[k][1] < j_longa[k+1][0] for k in range(len(j_longa)-1))